# 📊 RAISE-26 NLP模型结论总结

## AI对人类行为影响的多模型分析报告

本notebook汇总了4个模型的分析结论，回答核心问题：**AI最能影响人类行为的哪个方面？**

In [ ]:
# 环境配置
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体和样式
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

# 定义颜色方案
COLORS = {
    'primary': '#3498db',
    'success': '#2ecc71', 
    'warning': '#f39c12',
    'danger': '#e74c3c',
    'info': '#9b59b6'
}

print("✓ 环境配置完成")

---
## 📈 模型1: TF-IDF + Logistic Regression (Baseline)

**模型公式**: $y_k = \sigma(W_k^T \cdot x_{\text{tfidf}} + b_k)$

**性能**: Test Micro-F1 = 0.9430, Macro-F1 = 0.9331

In [ ]:
# 模型1结论数据
print("=" * 60)
print("📈 模型1: TF-IDF + Logistic Regression (Baseline)")
print("=" * 60)

# 性能指标
baseline_metrics = {
    '指标': ['Micro-F1', 'Macro-F1', 'Weighted-F1', 'Samples-F1'],
    'Validation': [0.9481, 0.9382, 0.9482, 0.9266],
    'Test': [0.9430, 0.9331, 0.9425, 0.9216]
}
df_metrics = pd.DataFrame(baseline_metrics)
print("\n📊 模型性能:")
print(df_metrics.to_string(index=False))

# 各类别F1分数
per_label_f1 = {
    'Label': [
        'Work, Jobs & Economy',
        'Learning, Knowledge & Education', 
        'Technology & Interaction',
        'Social Interaction & Relationships',
        'Routine, Lifestyle & Behavior',
        'Human Roles',
        'Creativity, Expression & Identity',
        'Society, Ethics & Culture',
        'Sentiment (Positive/Negative)',
        'Health, Safety & Risk',
        'Emotion, Motivation & Well-being',
        'Cognitive & Decision-Making'
    ],
    'F1': [0.982, 0.971, 0.962, 0.959, 0.957, 0.948, 0.943, 0.943, 0.931, 0.926, 0.889, 0.848],
    'Precision': [1.000, 0.984, 0.990, 0.986, 0.985, 0.975, 1.000, 0.982, 1.000, 0.955, 1.000, 0.807],
    'Recall': [0.965, 0.959, 0.936, 0.934, 0.930, 0.922, 0.892, 0.906, 0.871, 0.898, 0.800, 0.893]
}
df_label_f1 = pd.DataFrame(per_label_f1)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图: F1分数排名
ax1 = axes[0]
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(df_label_f1)))
bars = ax1.barh(df_label_f1['Label'], df_label_f1['F1'], color=colors)
ax1.set_xlabel('F1 Score', fontsize=12)
ax1.set_title('模型1: 各类别F1分数排名', fontsize=14, fontweight='bold')
ax1.set_xlim(0.8, 1.0)
ax1.axvline(x=0.9, color='red', linestyle='--', alpha=0.7, label='F1=0.9 基准线')

# 添加数值标签
for bar, val in zip(bars, df_label_f1['F1']):
    ax1.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', 
             va='center', fontsize=9)

ax1.legend()
ax1.invert_yaxis()

# 右图: 最好vs最差类别对比
ax2 = axes[1]
compare_labels = ['Work, Jobs & Economy\n(最佳)', 'Cognitive & Decision-Making\n(最差)']
compare_f1 = [0.982, 0.848]
compare_colors = ['#2ecc71', '#e74c3c']

bars2 = ax2.bar(compare_labels, compare_f1, color=compare_colors, width=0.5)
ax2.set_ylabel('F1 Score', fontsize=12)
ax2.set_title('最佳 vs 最差分类类别', fontsize=14, fontweight='bold')
ax2.set_ylim(0.7, 1.05)

for bar, val in zip(bars2, compare_f1):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}', 
             ha='center', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🔍 关键发现:")
print("   ✓ 'Work, Jobs & Economy' 分类最准确 (F1=0.982)")
print("     → 就业相关词汇(jobs, automation, layoffs)有明显词汇特征")
print("   ✗ 'Cognitive & Decision-Making' 分类最难 (F1=0.848)")
print("     → 与其他类别存在语义重叠，边界模糊")

---
## 🤖 模型2: DistilBERT (深度学习)

**架构**: DistilBERT [CLS] → Linear(768→256) → ReLU → Dropout → Linear(256→12) → Sigmoid

**参数量**: ~66M

In [ ]:
print("=" * 60)
print("🤖 模型2: DistilBERT (深度学习)")
print("=" * 60)

# 模型对比
model_comparison = {
    '模型': ['TF-IDF + LR (Baseline)', 'DistilBERT'],
    '参数量': ['~60K', '~66M'],
    '训练时间': ['< 1分钟', '~30分钟 (GPU)'],
    'Test Micro-F1': [0.9430, '~0.89'],
    'Test Macro-F1': [0.9331, '~0.87'],
    '可解释性': ['高 (词汇权重)', '低 (黑盒)']
}

df_comparison = pd.DataFrame(model_comparison)
print("\n📊 Baseline vs DistilBERT 对比:")
print(df_comparison.to_string(index=False))

# 可视化对比
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(2)
width = 0.35

baseline_scores = [0.9430, 0.9331]
bert_scores = [0.89, 0.87]

bars1 = ax.bar(x - width/2, baseline_scores, width, label='TF-IDF + LR', color='#3498db')
bars2 = ax.bar(x + width/2, bert_scores, width, label='DistilBERT', color='#9b59b6')

ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('模型性能对比: 简单模型 vs 深度学习', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['Micro-F1', 'Macro-F1'])
ax.legend()
ax.set_ylim(0.8, 1.0)

# 添加数值
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.2f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

print("\n💡 结论:")
print("   Baseline模型在此任务上表现已经很好!")
print("   → AI新闻标题的行为分类主要依赖【显式词汇特征】")
print("   → 而非需要深度学习才能捕获的【复杂语义关系】")
print("   → 简单模型 + 好的特征工程 可能比复杂模型更实用")

---
## 📚 模型3: NMF 主题模型

**数学公式**: $V \approx W \times H$

发现文本中的10个潜在主题

In [ ]:
print("=" * 60)
print("📚 模型3: NMF 主题模型")
print("=" * 60)

# NMF发现的10个主题
topics = {
    'Topic': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    '关键词': [
        'artificial intelligence, prediction, stock, technology',
        'ai, innovation, future, governance, data',
        'using ai, guide, 2025, complete guide',
        'use ai, work, health, ai use',
        'ai adoption, report, government, agentic ai',
        'new ai, study finds, tool, new study',
        'ai powered, launches, platform, digital assistant',
        'generative ai, learning, education, machine learning',
        'job, job market, job cuts, coming',
        'chatbot, musk, elon musk, grok'
    ],
    '主题解读': [
        'AI技术应用',
        '通用创新框架 ⭐主导',
        'AI使用指南',
        '工作健康应用',
        'AI政策采纳',
        'AI研究发现',
        'AI产品发布',
        '教育与学习',
        '就业影响 ⭐重要',
        '名人与聊天机器人'
    ],
    '文档占比(%)': [13.8, 52.4, 4.2, 5.1, 3.8, 4.5, 4.1, 5.3, 4.0, 2.8]
}

df_topics = pd.DataFrame(topics)
print("\n📊 发现的10个潜在主题:")
print(df_topics.to_string(index=False))

# 可视化主题分布
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图: 主题分布饼图
ax1 = axes[0]
sizes = df_topics['文档占比(%)']
labels = [f"T{i}" for i in range(10)]
colors = plt.cm.Set3(np.linspace(0, 1, 10))
explode = [0.1 if i == 1 else 0 for i in range(10)]  # 突出显示主导主题

wedges, texts, autotexts = ax1.pie(sizes, labels=labels, autopct='%1.1f%%',
                                   colors=colors, explode=explode, startangle=90)
ax1.set_title('主题分布 (Topic 1 主导: 52.4%)', fontsize=14, fontweight='bold')

# 右图: 主题条形图
ax2 = axes[1]
topic_labels = ['T1:创新框架', 'T0:AI技术', 'T7:教育学习', 'T3:工作健康', 
                'T5:研究发现', 'T2:使用指南', 'T6:产品发布', 'T8:就业影响',
                'T4:政策采纳', 'T9:名人聊天']
sorted_idx = np.argsort(sizes)[::-1]
sorted_sizes = [sizes[i] for i in sorted_idx]
sorted_labels = [topic_labels[i] for i in sorted_idx]
sorted_colors = [colors[i] for i in sorted_idx]

bars = ax2.barh(sorted_labels, sorted_sizes, color=sorted_colors)
ax2.set_xlabel('文档占比 (%)', fontsize=12)
ax2.set_title('各主题文档占比排名', fontsize=14, fontweight='bold')
ax2.invert_yaxis()

for bar, val in zip(bars, sorted_sizes):
    ax2.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
             va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\n💡 结论:")
print("   ⚠️ 52.4%的文档属于'通用创新框架'主题")
print("   → 媒体报道主要采用【泛泛的创新叙事】")
print("   → 具体行为影响(就业4%、教育5.3%、健康5.1%)只占较小比例")
print("   → 公众接收到的AI信息可能过于抽象，缺乏具体行为指导")

---
## 🔬 模型4: KMeans聚类 + LLM对比分析

**数学公式**: $\arg\min_C \sum_{i=1}^{k} \sum_{x \in C_i} ||x - \mu_i||^2$

对比 Mistral, Qwen, Llama 三个LLM生成的内容

In [ ]:
print("=" * 60)
print("🔬 模型4: KMeans聚类 + LLM对比分析")
print("=" * 60)

# LLM对比数据
llm_data = {
    'LLM': ['Mistral', 'Qwen', 'Llama'],
    '样本数': [2939, 2940, 2940],
    '平均标签数': [1.56, 1.52, 1.64],
    '熵(多样性)': [2.141, 2.069, 2.115],
    '主导类别': ['Cognitive (23.2%)', 'Cognitive (20.6%)', 'Cognitive (20.2%)']
}

df_llm = pd.DataFrame(llm_data)
print("\n📊 三个LLM生成内容对比:")
print(df_llm.to_string(index=False))

# 各LLM的Top5标签分布
llm_labels = {
    'Mistral': {
        'Cognitive & Decision-Making': 23.2,
        'Routine, Lifestyle & Behavior': 15.5,
        'Creativity, Expression & Identity': 8.4,
        'Social Interaction': 7.8,
        'Learning & Education': 7.8
    },
    'Qwen': {
        'Cognitive & Decision-Making': 20.6,
        'Routine, Lifestyle & Behavior': 15.8,
        'Sentiment': 8.6,
        'Society, Ethics & Culture': 8.3,
        'Social Interaction': 8.0
    },
    'Llama': {
        'Cognitive & Decision-Making': 20.2,
        'Routine, Lifestyle & Behavior': 16.5,
        'Creativity, Expression & Identity': 8.7,
        'Society, Ethics & Culture': 8.6,
        'Sentiment': 8.3
    }
}

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 左上: 熵值对比
ax1 = axes[0, 0]
entropy_colors = ['#e74c3c', '#3498db', '#2ecc71']
bars1 = ax1.bar(df_llm['LLM'], df_llm['熵(多样性)'], color=entropy_colors)
ax1.set_ylabel('信息熵', fontsize=12)
ax1.set_title('主题多样性对比 (熵值越高越多样)', fontsize=14, fontweight='bold')
ax1.axhline(y=np.median(df_llm['熵(多样性)']), color='gray', linestyle='--', label='中位数')
ax1.legend()
for bar, val in zip(bars1, df_llm['熵(多样性)']):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}',
             ha='center', fontsize=11, fontweight='bold')

# 右上: 平均标签数
ax2 = axes[0, 1]
bars2 = ax2.bar(df_llm['LLM'], df_llm['平均标签数'], color=entropy_colors)
ax2.set_ylabel('平均标签数', fontsize=12)
ax2.set_title('输出多维度性对比', fontsize=14, fontweight='bold')
for bar, val in zip(bars2, df_llm['平均标签数']):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.2f}',
             ha='center', fontsize=11, fontweight='bold')

# 左下: 各LLM的Top标签堆叠图
ax3 = axes[1, 0]
categories = ['Cognitive', 'Routine', 'Creativity', 'Social', 'Sentiment', 'Ethics', 'Education']
mistral_vals = [23.2, 15.5, 8.4, 7.8, 0, 0, 7.8]
qwen_vals = [20.6, 15.8, 0, 8.0, 8.6, 8.3, 0]
llama_vals = [20.2, 16.5, 8.7, 0, 8.3, 8.6, 0]

x = np.arange(len(categories))
width = 0.25

ax3.bar(x - width, mistral_vals, width, label='Mistral', color='#e74c3c')
ax3.bar(x, qwen_vals, width, label='Qwen', color='#3498db')
ax3.bar(x + width, llama_vals, width, label='Llama', color='#2ecc71')

ax3.set_ylabel('占比 (%)', fontsize=12)
ax3.set_title('各LLM主要行为类别分布', fontsize=14, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(categories, rotation=45, ha='right')
ax3.legend()

# 右下: 统计检验结果
ax4 = axes[1, 1]
ax4.axis('off')
stats_text = """
📊 统计检验结果
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[1] 卡方检验 (Chi-Square Test)
    χ² = 74.21
    df = 22
    p-value = 1.41e-07 ✓
    
    结论: p < 0.05, 拒绝原假设
    → LLM之间存在显著差异!

[2] 效应量 (Cramér's V)
    V = 0.065 (小效应)
    
    结论: 虽然差异显著，
    但实际影响程度较小

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💡 所有LLM都以"认知与决策"为主导!
"""
ax4.text(0.1, 0.5, stats_text, transform=ax4.transAxes, fontsize=12,
         verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n💡 结论:")
print("   ✓ 所有LLM都以 'Cognitive & Decision-Making' 为主导 (20-23%)")
print("   ✓ Mistral 输出最多样 (熵=2.141)")
print("   ✓ Llama 输出最多维 (平均1.64个标签)")
print("   ⚠️ LLM生成内容强调AI对【思维和决策】的影响")

---
## 🎯 核心问题: AI最能影响哪个方面?

In [ ]:
print("=" * 60)
print("🎯 核心问题: AI最能影响人类行为的哪个方面?")
print("=" * 60)

# 综合数据
impact_ranking = {
    '排名': ['🥇', '🥈', '🥉', '4', '5', '6', '7', '8', '9', '10', '11', '12'],
    '行为领域': [
        'Work, Jobs & Economy',
        'Learning, Knowledge & Education',
        'Technology & Interaction',
        'Society, Ethics & Culture',
        'Routine, Lifestyle & Behavior',
        'Sentiment (Positive/Negative)',
        'Human Roles',
        'Health, Safety & Risk',
        'Creativity, Expression & Identity',
        'Cognitive & Decision-Making',
        'Social Interaction & Relationships',
        'Emotion, Motivation & Well-being'
    ],
    '样本数': [2526, 1946, 1733, 1584, 1479, 1441, 1256, 1230, 877, 832, 730, 551],
    '占比(%)': [24.1, 18.5, 16.5, 15.1, 14.1, 13.7, 12.0, 11.7, 8.4, 7.9, 7.0, 5.2]
}

df_impact = pd.DataFrame(impact_ranking)
print("\n📊 AI影响力排名 (基于媒体报道频率):")
print(df_impact.to_string(index=False))

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 左图: 完整排名
ax1 = axes[0]
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, 12))
bars = ax1.barh(df_impact['行为领域'], df_impact['占比(%)'], color=colors)
ax1.set_xlabel('占比 (%)', fontsize=12)
ax1.set_title('AI影响人类行为的12个领域\n(按媒体报道频率排名)', fontsize=14, fontweight='bold')
ax1.invert_yaxis()

# 添加数值和奖牌
medals = ['🥇', '🥈', '🥉'] + [''] * 9
for i, (bar, val, medal) in enumerate(zip(bars, df_impact['占比(%)'], medals)):
    ax1.text(val + 0.3, bar.get_y() + bar.get_height()/2, 
             f'{medal} {val:.1f}%', va='center', fontsize=10)

# 右图: Top 5 饼图
ax2 = axes[1]
top5_labels = df_impact['行为领域'][:5].tolist()
top5_sizes = df_impact['占比(%)'][:5].tolist()
top5_sizes.append(100 - sum(top5_sizes))  # 其他
top5_labels.append('其他 (7个类别)')

colors_pie = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#95a5a6']
explode = [0.05, 0, 0, 0, 0, 0]

wedges, texts, autotexts = ax2.pie(top5_sizes, labels=top5_labels, autopct='%1.1f%%',
                                   colors=colors_pie, explode=explode, startangle=90)
ax2.set_title('Top 5 AI影响领域', fontsize=14, fontweight='bold')

# 调整标签字体
for text in texts:
    text.set_fontsize(9)

plt.tight_layout()
plt.show()

In [ ]:
# 媒体视角 vs LLM视角 对比
print("\n" + "=" * 60)
print("📰 媒体视角 vs 🤖 LLM视角 对比")
print("=" * 60)

comparison = {
    '排名': [1, 2, 3],
    '媒体报道视角': [
        'Work, Jobs & Economy (24.1%)',
        'Learning & Education (18.5%)',
        'Technology & Interaction (16.5%)'
    ],
    'LLM生成内容视角': [
        'Cognitive & Decision-Making (20-23%)',
        'Routine, Lifestyle & Behavior (15-16%)',
        'Creativity & Identity / Sentiment (8-9%)'
    ]
}

df_vs = pd.DataFrame(comparison)
print(df_vs.to_string(index=False))

# 可视化对比
fig, ax = plt.subplots(figsize=(12, 6))

categories = ['第1位', '第2位', '第3位']
media_vals = [24.1, 18.5, 16.5]
llm_vals = [21.3, 15.9, 8.5]  # 平均值

x = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x - width/2, media_vals, width, label='📰 媒体视角', color='#3498db')
bars2 = ax.bar(x + width/2, llm_vals, width, label='🤖 LLM视角', color='#e74c3c')

ax.set_ylabel('占比 (%)', fontsize=12)
ax.set_title('媒体 vs LLM: AI影响领域的不同视角', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()

# 添加标签
media_labels = ['Work/Jobs', 'Education', 'Technology']
llm_labels = ['Cognitive', 'Routine', 'Creativity']

for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
    ax.text(bar1.get_x() + bar1.get_width()/2, bar1.get_height() + 0.5,
            media_labels[i], ha='center', fontsize=9, rotation=45)
    ax.text(bar2.get_x() + bar2.get_width()/2, bar2.get_height() + 0.5,
            llm_labels[i], ha='center', fontsize=9, rotation=45)

plt.tight_layout()
plt.show()

---
## 📝 最终结论

In [ ]:
print("\n" + "=" * 70)
print("📝 最终结论: AI最能影响人类行为的哪个方面?")
print("=" * 70)

conclusions = """
┌─────────────────────────────────────────────────────────────────────┐
│                        🎯 核心发现                                  │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  📰 媒体视角 (Dataset A: 10,500条新闻标题)                          │
│  ─────────────────────────────────────────                          │
│  AI最影响: "工作与经济" > "教育学习" > "技术交互"                   │
│                                                                     │
│  → 公众媒体主要从【经济和功能性】角度报道AI                         │
│  → 强调: 工作、生产力、技能、就业                                   │
│                                                                     │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  🤖 LLM视角 (Dataset C: 8,820条LLM生成内容)                         │
│  ─────────────────────────────────────────                          │
│  AI最影响: "认知与决策" > "日常行为" > "创造力与身份"               │
│                                                                     │
│  → LLM更强调AI对【思维方式和决策过程】的影响                        │
│  → 强调: 思考、判断、推理、选择                                     │
│                                                                     │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ⚠️ 被忽视的领域                                                    │
│  ─────────────────                                                  │
│  • 情感与幸福感 (Emotion & Well-being): 仅占 5.2%                   │
│  • 社交互动 (Social Interaction): 仅占 7.0%                         │
│                                                                     │
│  → 媒体和AI系统都相对忽视AI对【情感和人际关系】的影响               │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘

💡 总结:
   AI正在全方位改变人类行为，但当前话语体系主要强调:
   
   ✓ 工作/就业 (媒体最关注)
   ✓ 认知/决策 (LLM最强调)
   
   而情感、人际关系等"软性"领域的影响被严重低估。
"""

print(conclusions)

# 最终可视化
fig, ax = plt.subplots(figsize=(10, 8))
ax.axis('off')

# 绘制总结图
summary_data = {
    'Work & Economy': (0.5, 0.85, 24.1, '#e74c3c'),
    'Education': (0.5, 0.70, 18.5, '#3498db'),
    'Technology': (0.5, 0.55, 16.5, '#2ecc71'),
    'Ethics & Culture': (0.5, 0.40, 15.1, '#f39c12'),
    'Routine & Behavior': (0.5, 0.25, 14.1, '#9b59b6'),
    'Emotion (被忽视)': (0.5, 0.10, 5.2, '#95a5a6')
}

for label, (x, y, size, color) in summary_data.items():
    circle = plt.Circle((x, y), size/100, color=color, alpha=0.7)
    ax.add_patch(circle)
    ax.text(x + 0.35, y, f"{label}\n({size}%)", fontsize=11, va='center')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('AI对人类行为影响的领域分布\n(圆圈大小代表关注度)', 
             fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n✅ 分析完成!")